# Import

In [ ]:
import logging
import sys
from pathlib import Path
import pandas as pd
from xgboost import XGBRegressor    

# Config

In [ ]:
# Google Driver Config
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = Path('/content/drive/MyDrive/colab_env/')
INPUT_PATH = DRIVE_PATH / 'dataset_china_all'
OUTPUT_PATH = DRIVE_PATH / 'model_warehouse'
FEATURE_PATH = INPUT_PATH / 'features'
MARKET_DATA_PATH = INPUT_PATH / 'market_data'

In [ ]:
# Model Config
START_YEAR = 2020
END_YEAR = 2026
RETRAIN_MONTH = 6
WINDOW_TYPE = "rolling"
WINDOW_SIZE = 24

BASE_MODEL_NAME = "xgb"
MODEL_PARAMS = {
    "n_estimators": 5800,
    "max_depth": 8,
    "learning_rate": 0.0001,
    "min_child_weight": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "alpha": 0.0,
    "lambda": 1.0,
    "gamma": 0.0,
    "device": "cuda",
}
LABEL_HORIZON = 5

# Functions

## logger

In [ ]:
def setup_logger(
    name: str = "q-modeling",
    log_file: str = "app.log",
    level: int = logging.INFO,
) -> logging.Logger:
    """
    Setup a logger with console and file handlers.

    Args:
        name: Name of the logger.
        log_file: Name of the log file.
        level: Logging level.

    Returns:
        Configured logger instance.
    """
    # Create logs directory if it doesn't exist
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)

    # Create logger
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Check if handlers are already added to avoid duplicates
    if not logger.handlers:
        # Create formatter
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )

        # Create file handler
        file_handler = logging.FileHandler(log_dir / log_file, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        # Create console handler
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

    return logger

# Create a default logger instance
logger = setup_logger()

## data ingest

In [ ]:
def load_label(start: pd.Timestamp, end: pd.Timestamp):
    filename = INPUT_PATH / 'labels.parquet'
    filters = [
        ("datetime", ">=", self.start),
        ("datetime", "<=", self.end),
    ]
    columns = [f'label_{LABEL_HORIZON}d']
    return pd.read_parquet(filename, filters=filters, columns=columns)


class FeatureLoader:
    def __init__(self, start: pd.Timestamp, end: pd.Timestamp):
        self.start = start
        self.end = end
        self.directory = FEATURE_PATH
        self._find_name_to_filename()

    def _find_name_to_filename(self):
        self.name_to_filename = {}
        for filename in self.directory.glob("*.parquet"):
            meta = pq.read_metadata(filename)
            name = meta.schema.names[-1]
            self.name_to_filename[name] = filename
        return self.name_to_filename

    def load_feature(self, name: str) -> pd.DataFrame:

        filters = [
            ("datetime", ">=", self.start),
            ("datetime", "<=", self.end),
        ]
        try:
            filename = self.name_to_filename[name]
            df = pd.read_parquet(
                filename,
                filters=filters,
            )
        except Exception:
            warnings.warn(f"Failed to load {name}, skip.")
            return pd.DataFrame()
        df = df.set_index(["datetime", "symbol"]).sort_index()

        return df

    def load_features(self, names: list[str] = None) -> pd.DataFrame:

        if names is None:
            names = list(self.name_to_filename.keys())

        logger.debug(f"Loading {len(names)} features")

        container = []
        for name in names:
            df = self.load_feature(name)
            container.append(df)

        if not container:
            return pd.DataFrame()

        return pd.concat(container, axis=1).sort_index()


# Pipeline